# 🏦 Credit Risk ML — Give Me Some Credit (Dataset Real)

> Pipeline completo com o dataset real do Kaggle  
> Logistic Regression · Random Forest · XGBoost · Simulação Financeira

---

**Dataset:** [Give Me Some Credit](https://www.kaggle.com/c/GiveMeSomeCredit)  
**Target:** `SeriousDlqin2yrs` — inadimplência grave (90+ dias de atraso)

| # | Seção |
|---|-------|
| 1 | Imports & Setup |
| 2 | Carregamento e Normalização de Colunas |
| 3 | EDA — Problemas Conhecidos do GMSC |
| 4 | Feature Engineering (específico GMSC) |
| 5 | Split + Normalização |
| 6 | Baseline — Logistic Regression |
| 7 | Random Forest |
| 8 | XGBoost |
| 9 | Comparação de Modelos |
| 10 | Simulação de Decisão |
| 11 | Matrizes de Confusão |
| 12 | Conclusão |

---

## 1. 📦 Imports & Setup

In [ ]:
import warnings, time
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import (
    roc_auc_score, roc_curve, auc,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score, accuracy_score,
    precision_recall_curve,
)
from xgboost import XGBClassifier

SEED      = 42
TEST_SIZE = 0.20
THRESHOLD = 0.70
np.random.seed(SEED)

plt.rcParams.update({
    'figure.facecolor':'#0F1117','axes.facecolor':'#161B22','axes.edgecolor':'#30363D',
    'axes.labelcolor':'#E6EDF3','xtick.color':'#8B949E','ytick.color':'#8B949E',
    'text.color':'#E6EDF3','grid.color':'#21262D','grid.linewidth':0.8,
    'figure.dpi':120,'font.family':'monospace',
})
C = {'lr':'#58A6FF','rf':'#3FB950','xgb':'#F78166','good':'#3FB950','bad':'#F78166',
     'accent':'#D2A8FF','warn':'#E3B341','muted':'#8B949E'}

print('✅  Setup OK')

---
## 2. 📂 Carregamento e Normalização de Colunas

O GMSC tem colunas com nomes longos e inconsistentes. O primeiro passo é renomear para um schema limpo e tratar os **problemas conhecidos** deste dataset específico.

| Coluna original | Schema interno | Problema conhecido |
|---|---|---|
| `SeriousDlqin2yrs` | `default` | TARGET |
| `MonthlyIncome` | `income` | ~20% nulos |
| `NumberOfDependents` | `num_dependents` | ~2.5% nulos |
| `RevolvingUtilizationOfUnsecuredLines` | `revolving_util` | valores > 1 (> 100%) |
| `DebtRatio` | `debt_ratio` | valores extremamente altos |
| `NumberOfTime*DaysPastDue*` | `late_30_59`, `late_60_89`, `late_90` | valores > 90 (erro) |
| `age` | `age` | alguns registros com age == 0 |

In [ ]:
# ── Mapeamento de colunas ─────────────────────────────────────────────────────
COLUMN_MAP = {
    'SeriousDlqin2yrs'                     : 'default',
    'RevolvingUtilizationOfUnsecuredLines'  : 'revolving_util',
    'age'                                   : 'age',
    'NumberOfTime30-59DaysPastDueNotWorse'  : 'late_30_59',
    'DebtRatio'                             : 'debt_ratio',
    'MonthlyIncome'                         : 'income',
    'NumberOfOpenCreditLinesAndLoans'       : 'num_open_accounts',
    'NumberOfTimes90DaysLate'               : 'late_90',
    'NumberRealEstateLoansOrLines'          : 'num_real_estate',
    'NumberOfTime60-89DaysPastDueNotWorse'  : 'late_60_89',
    'NumberOfDependents'                    : 'num_dependents',
}

# ── Carregamento ──────────────────────────────────────────────────────────────
RAW_PATH = '/kaggle/input/GiveMeSomeCredit/cs-training.csv'
# RAW_PATH = 'data/raw/give_me_some_credit.csv'  # local

df_raw = pd.read_csv(RAW_PATH)
# Remove coluna de índice do Kaggle se existir
df_raw = df_raw.loc[:, ~df_raw.columns.str.startswith('Unnamed')]
df_raw.rename(columns=COLUMN_MAP, inplace=True)

print(f'✅  Dataset carregado | shape={df_raw.shape}')
display(df_raw.head())

In [ ]:
# ── Limpeza específica do GMSC ────────────────────────────────────────────────
df = df_raw.copy()

# 1. Target
df.dropna(subset=['default'], inplace=True)
df['default'] = df['default'].astype(int)

# 2. income — ~20% nulos → mediana
income_median = df['income'].median()
n_income_null = df['income'].isna().sum()
df['income'].fillna(income_median, inplace=True)
print(f'  income : {n_income_null:,} nulos imputados pela mediana ({income_median:,.0f})')

# 3. num_dependents — nulos → 0 (moda e interpretação natural)
n_dep_null = df['num_dependents'].isna().sum()
df['num_dependents'].fillna(0, inplace=True)
print(f'  num_dependents: {n_dep_null:,} nulos imputados com 0')

# 4. age == 0 — registros inválidos
n_age_zero = (df['age'] == 0).sum()
df = df[df['age'] > 0].copy()
print(f'  age==0 : {n_age_zero} registros removidos')

# 5. Clipping de outliers documentados
df['revolving_util'] = df['revolving_util'].clip(0, 1.5)    # >100% possível, >150% é erro
df['debt_ratio']     = df['debt_ratio'].clip(0, 3.0)         # >3× a renda é ruído
for col in ['late_30_59', 'late_60_89', 'late_90']:
    df[col] = df[col].clip(0, 20)                             # >20 é erro de digitação
df['income'] = df['income'].clip(0, 500_000)
df['age']    = df['age'].clip(18, 100)

# Tipos
for col in ['age','num_open_accounts','num_dependents','late_30_59','late_60_89','late_90','num_real_estate']:
    df[col] = df[col].astype(int)

print(f'\n✅  Limpeza concluída')
print(f'   Shape final     : {df.shape}')
print(f'   Nulos restantes : {df.isnull().sum().sum()}')
print(f'   Taxa de default : {df["default"].mean():.1%}')

---
## 3. 🔍 EDA — Análise Exploratória

In [ ]:
# ── 3.1 Target ────────────────────────────────────────────────────────────────
counts = df['default'].value_counts()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.patch.set_facecolor('#0F1117')

ax = axes[0]
bars = ax.bar(['Adimplente (0)','Inadimplente (1)'], counts.values,
              color=[C['good'],C['bad']], edgecolor='#0F1117', width=0.5)
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
            f'{v:,}', ha='center', fontsize=12, fontweight='bold')
ax.set_title('Contagem por Classe', color='#E6EDF3', fontsize=12)
ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
ax2.pie(counts.values, labels=['Adimplente','Inadimplente'],
        colors=[C['good'],C['bad']], autopct='%1.1f%%', startangle=90,
        pctdistance=0.7, wedgeprops={'edgecolor':'#0F1117','linewidth':3})
ax2.set_title('Proporção (GMSC: ~6.7%)', color='#E6EDF3', fontsize=12)

ax3 = axes[2]
nulls_orig = df_raw.isnull().sum().sort_values(ascending=False)
nulls_orig = nulls_orig[nulls_orig > 0]
ax3.barh(nulls_orig.index, nulls_orig.values, color=C['warn'], edgecolor='#0F1117', alpha=0.85)
ax3.set_title('Nulos no Dataset Original', color='#E6EDF3', fontsize=12)
ax3.set_xlabel('Quantidade'); ax3.grid(axis='x', alpha=0.3)

plt.suptitle('GMSC — Target e Qualidade dos Dados',
             fontsize=14, color=C['accent'], fontweight='bold')
plt.tight_layout(); plt.show()

ratio = counts[0] / counts[1]
print(f'  Taxa de default : {df["default"].mean():.1%}')
print(f'  Ratio adim:inad  = {ratio:.1f}:1  → scale_pos_weight = {ratio:.1f}')

In [ ]:
# ── 3.2 Distribuições das features por classe ─────────────────────────────────
num_cols = ['age', 'income', 'revolving_util', 'debt_ratio',
            'late_30_59', 'late_60_89', 'late_90', 'num_open_accounts']

fig, axes = plt.subplots(2, 4, figsize=(17, 7))
fig.patch.set_facecolor('#0F1117')
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    for val, color, label in [(0,C['good'],'Adimplente'),(1,C['bad'],'Inadimplente')]:
        subset = df[df['default']==val][col]
        ax.hist(subset, bins=40, alpha=0.55, color=color, label=label, density=True)
    corr_v = df[col].corr(df['default'])
    ax.set_title(f'{col}\n(corr={corr_v:+.3f})', fontsize=9, color='#E6EDF3')
    ax.legend(fontsize=7, framealpha=0.2); ax.grid(alpha=0.2)

plt.suptitle('Distribuições por Classe de Default — GMSC',
             fontsize=14, color=C['accent'], fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.3 Correlação ────────────────────────────────────────────────────────────
corr = df.corr()
target_corr = corr['default'].drop('default').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor('#0F1117')

# Heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap=sns.diverging_palette(220,20,as_cmap=True),
            vmax=0.6, vmin=-0.6, center=0, annot=True, fmt='.2f',
            annot_kws={'size':8}, square=True, linewidths=0.4,
            linecolor='#0F1117', ax=axes[0])
axes[0].set_title('Correlação GMSC', fontsize=13, color='#E6EDF3')

# Ranking com target
colors_bar = [C['bad'] if v > 0 else C['good'] for v in target_corr.values]
axes[1].barh(target_corr.index, target_corr.values,
             color=colors_bar, alpha=0.85, edgecolor='#0F1117')
axes[1].axvline(0, color='#8B949E', lw=1)
axes[1].set_title('Correlação com Default (GMSC)',
                  fontsize=13, color='#E6EDF3')
axes[1].grid(axis='x', alpha=0.25)

plt.tight_layout(); plt.show()

print('  Top 3 positivas (↑ risco):', target_corr.nlargest(3).index.tolist())
print('  Top 3 negativas (↓ risco):', target_corr.nsmallest(3).index.tolist())

---
## 4. ⚙️ Feature Engineering (específico GMSC)

As features derivadas são diferentes das usadas no dataset sintético, pois o GMSC não tem `credit_score`, `loan_amount` nem `employment_years`.
O risco aqui é capturado principalmente pelo **histórico de atrasos** e pela **utilização de crédito rotativo**.

| Nova Feature | Fórmula | Intuição |
|---|---|---|
| `total_late` | `late_30 + late_60 + late_90` | Volume total de inadimplências |
| `late_severity` | `late_90 / total_late` | Proporção de atrasos graves |
| `util_x_debt` | `revolving_util × debt_ratio` | Pressão dupla de crédito |
| `income_per_dep` | `income / (dependents + 1)` | Capacidade real de pagamento |
| `high_util_flag` | `revolving_util > 0.75` | Flag de utilização crítica |
| `multi_late_flag` | atrasos em > 1 bucket | Padrão de inadimplência |
| `log_income` | `log1p(income)` | Normaliza a distribuição log-normal |

In [ ]:
def build_features_gmsc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['total_late']    = (df['late_30_59'] + df['late_60_89'] + df['late_90']).astype(float)

    df['late_severity'] = np.where(
        df['total_late'] > 0, df['late_90'] / df['total_late'], 0.0
    ).round(4)

    df['util_x_debt']   = (df['revolving_util'].clip(0,1.5) * df['debt_ratio'].clip(0,3.0)).round(6)

    df['income_per_dep'] = (df['income'] / (df['num_dependents'] + 1)).round(2)

    df['high_util_flag'] = (df['revolving_util'] > 0.75).astype(int)

    late_buckets        = ((df['late_30_59']>0).astype(int)
                          +(df['late_60_89']>0).astype(int)
                          +(df['late_90']>0).astype(int))
    df['multi_late_flag'] = (late_buckets > 1).astype(int)

    df['log_income']    = np.log1p(df['income']).round(4)

    return df


df_fe    = build_features_gmsc(df)
new_cols = ['total_late','late_severity','util_x_debt',
            'income_per_dep','high_util_flag','multi_late_flag','log_income']

print('✅  Features criadas')
print(f'   Base: {len([c for c in df_fe.columns if c not in new_cols+["default"]])} | '
      f'Novas: {len(new_cols)} | Total: {len(df_fe.columns)-1}')

print('\n   Correlação com default:')
for col in new_cols:
    corr_v = df_fe[col].corr(df_fe['default'])
    bar    = '█' * int(abs(corr_v)*50)
    print(f'   {col:<20} {('+' if corr_v>0 else '-')}{bar:<40} {corr_v:+.4f}')

In [ ]:
# Visualização das novas features
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
fig.patch.set_facecolor('#0F1117')
axes = axes.flatten()

for i, col in enumerate(new_cols):
    ax = axes[i]
    if df_fe[col].nunique() <= 2:
        rates = df_fe.groupby(col)['default'].mean()
        ax.bar(rates.index.astype(str), rates.values,
               color=[C['good'],C['bad']][:len(rates)], edgecolor='#0F1117', width=0.4)
        ax.set_ylabel('Taxa de Default')
    else:
        for val, color, label in [(0,C['good'],'Adimplente'),(1,C['bad'],'Inadimplente')]:
            ax.hist(df_fe[df_fe['default']==val][col], bins=40,
                    alpha=0.6, color=color, label=label, density=True)
        ax.legend(fontsize=7, framealpha=0.2)
    corr_v = df_fe[col].corr(df_fe['default'])
    ax.set_title(f'{col}\n(r={corr_v:+.3f})', fontsize=9, color='#E6EDF3')
    ax.grid(alpha=0.2)

axes[-1].set_visible(False)
plt.suptitle('Novas Features × Default — GMSC',
             fontsize=14, color=C['accent'], fontweight='bold')
plt.tight_layout(); plt.show()

---
## 5. ✂️ Split + Normalização

In [ ]:
TARGET   = 'default'
FEATURES = [c for c in df_fe.columns if c != TARGET]

X = df_fe[FEATURES]
y = df_fe[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
assert abs(y_train.mean()-y_test.mean()) < 0.005, 'Estratificação falhou'

scaler    = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train),  columns=FEATURES, index=X_train.index)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),       columns=FEATURES, index=X_test.index)

n_neg = (y_train==0).sum()
n_pos = (y_train==1).sum()
SPW   = n_neg / n_pos

print(f'  Treino  : {len(X_train):,}  (default={y_train.mean():.1%})')
print(f'  Teste   : {len(X_test):,}  (default={y_test.mean():.1%})')
print(f'  Features: {len(FEATURES)}')
print(f'  scale_pos_weight = {SPW:.2f}')
print('\n✅  Split e normalização OK (zero data leakage)')

In [ ]:
def evaluate_model(model, X_test, y_test, name, threshold=0.5, verbose=True):
    y_prob = model.predict_proba(X_test)[:,1]
    y_pred = (y_prob >= threshold).astype(int)
    m = {'name':name, 'y_prob':y_prob, 'y_pred':y_pred, 'model':model,
         'auc':roc_auc_score(y_test,y_prob), 'accuracy':accuracy_score(y_test,y_pred),
         'precision':precision_score(y_test,y_pred,zero_division=0),
         'recall':recall_score(y_test,y_pred,zero_division=0),
         'f1':f1_score(y_test,y_pred,zero_division=0)}
    if verbose:
        print(f'\n{'─'*50}\n  {name}\n{'─'*50}')
        for k in ['auc','accuracy','precision','recall','f1']:
            print(f'  {k:<12}: {m[k]:.4f}')
    return m

---
## 6. 📏 Baseline — Logistic Regression

In [ ]:
t0 = time.time()
lr_model = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced',
                               solver='lbfgs', random_state=SEED, n_jobs=-1)
lr_model.fit(X_train_sc, y_train)
lr_res = evaluate_model(lr_model, X_test_sc, y_test, 'Logistic Regression')
print(f'\n  Tempo: {time.time()-t0:.2f}s | Iterações: {lr_model.n_iter_[0]}')
print(f'\n{classification_report(y_test, lr_res["y_pred"], target_names=["Adimplente","Inadimplente"])}')

---
## 7. 🌲 Random Forest

In [ ]:
t0 = time.time()
rf_model = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=10,
                                   max_features='sqrt', class_weight='balanced',
                                   random_state=SEED, n_jobs=-1)
rf_model.fit(X_train_sc, y_train)
rf_res = evaluate_model(rf_model, X_test_sc, y_test, 'Random Forest')
print(f'\n  Tempo: {time.time()-t0:.1f}s | Δ vs baseline: {rf_res["auc"]-lr_res["auc"]:+.4f}')

imp_rf = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9,6)); fig.patch.set_facecolor('#0F1117')
colors_bar = [C['accent'] if v > imp_rf.quantile(0.75) else C['rf'] for v in imp_rf.values]
ax.barh(imp_rf.index, imp_rf.values, color=colors_bar, edgecolor='#0F1117', alpha=0.9)
ax.set_title('Feature Importance — Random Forest (GMSC)', fontsize=13, fontweight='bold', color='#E6EDF3')
ax.grid(axis='x', alpha=0.25); plt.tight_layout(); plt.show()

---
## 8. ⚡ XGBoost

In [ ]:
t0 = time.time()
xgb_model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8,
                           reg_alpha=0.1, reg_lambda=1.0,
                           scale_pos_weight=SPW,
                           eval_metric='auc', early_stopping_rounds=30,
                           random_state=SEED, n_jobs=-1, verbosity=0)
xgb_model.fit(X_train_sc, y_train, eval_set=[(X_test_sc,y_test)], verbose=False)
xgb_res = evaluate_model(xgb_model, X_test_sc, y_test, 'XGBoost')
print(f'\n  Tempo: {time.time()-t0:.1f}s | Best iter: {xgb_model.best_iteration} | Δ vs baseline: {xgb_res["auc"]-lr_res["auc"]:+.4f}')

imp_xgb = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9,6)); fig.patch.set_facecolor('#0F1117')
colors_bar = [C['accent'] if v > imp_xgb.quantile(0.75) else C['xgb'] for v in imp_xgb.values]
ax.barh(imp_xgb.index, imp_xgb.values, color=colors_bar, edgecolor='#0F1117', alpha=0.9)
ax.set_title('Feature Importance — XGBoost (GMSC)', fontsize=13, fontweight='bold', color='#E6EDF3')
ax.grid(axis='x', alpha=0.25); plt.tight_layout(); plt.show()

---
## 9. 📊 Comparação de Modelos

In [ ]:
all_results  = [lr_res, rf_res, xgb_res]
model_colors = {'Logistic Regression':C['lr'],'Random Forest':C['rf'],'XGBoost':C['xgb']}

df_comp = pd.DataFrame([
    {k:v for k,v in r.items() if k not in ('y_prob','y_pred','model')}
    for r in all_results
]).set_index('name').round(4)

print(df_comp.to_string())
best_name = df_comp['auc'].idxmax()
best_res  = next(r for r in all_results if r['name']==best_name)
print(f'\n🏆  Melhor: {best_name}  (AUC={df_comp.loc[best_name,"auc"]:.4f})')

# Curvas ROC + Barras
fig = plt.figure(figsize=(14,5)); fig.patch.set_facecolor('#0F1117')
gs  = gridspec.GridSpec(1,2,figure=fig,wspace=0.35)

ax1 = fig.add_subplot(gs[0])
ax1.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
for r in all_results:
    fpr,tpr,_ = roc_curve(y_test, r['y_prob'])
    ax1.plot(fpr,tpr,lw=2.2,color=model_colors[r['name']],
             label=f'{r["name"]} ({r["auc"]:.3f})')
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('Curvas ROC — GMSC',fontsize=13,fontweight='bold',color='#E6EDF3')
ax1.legend(fontsize=9,framealpha=0.3); ax1.grid(alpha=0.2)

ax2 = fig.add_subplot(gs[1])
metrics_to_plot = ['auc','f1','precision','recall']
x,w = np.arange(len(metrics_to_plot)), 0.25
for i,r in enumerate(all_results):
    vals = [r[m] for m in metrics_to_plot]
    bars = ax2.bar(x+i*w,vals,w,label=r['name'],color=model_colors[r['name']],alpha=0.9,edgecolor='#0F1117')
    for bar,v in zip(bars,vals):
        ax2.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005,f'{v:.2f}',ha='center',fontsize=7.5)
ax2.set_xticks(x+w); ax2.set_xticklabels([m.upper() for m in metrics_to_plot],fontsize=9)
ax2.set_ylim(0,1.12)
ax2.set_title('Métricas',fontsize=13,fontweight='bold',color='#E6EDF3')
ax2.legend(fontsize=9,framealpha=0.3); ax2.grid(axis='y',alpha=0.2)

plt.suptitle('Comparação de Modelos — GMSC',fontsize=15,color=C['accent'],fontweight='bold')
plt.tight_layout(); plt.show()

---
## 10. 💰 Simulação de Decisão

In [ ]:
PROFIT_GOOD = 100; LOSS_BAD = 500; OPP_COST = 10

def simulate_business(prob, y_true, threshold=THRESHOLD,
                       profit=PROFIT_GOOD, loss=LOSS_BAD, opp=OPP_COST):
    approved = prob <= threshold
    y = np.array(y_true)
    tn = int(np.sum( approved&(y==0)))
    fn = int(np.sum( approved&(y==1)))
    fp = int(np.sum(~approved&(y==0)))
    tp = int(np.sum(~approved&(y==1)))
    lucro = tn*profit - fn*loss - fp*opp
    return {'tn':tn,'fn':fn,'fp':fp,'tp':tp,
            'lucro':lucro,'aprovacao':float(approved.mean())}


for r in all_results:
    s    = simulate_business(r['y_prob'], y_test)
    sign = '+' if s['lucro']>=0 else ''
    print(f"  {r['name']:<22} aprovação={s['aprovacao']:.1%}  "
          f"✅{s['tn']:>6,}  ❌{s['fn']:>5,}  lucro={sign}R${s['lucro']:>10,.0f}")

# Otimização
best_prob  = best_res['y_prob']
thresholds = np.linspace(0.01,0.99,200)
profits    = [simulate_business(best_prob,y_test,t)['lucro'] for t in thresholds]
best_t     = thresholds[np.argmax(profits)]

fig,ax = plt.subplots(figsize=(10,4)); fig.patch.set_facecolor('#0F1117')
ax.plot(thresholds,np.array(profits)/1000,lw=2.5,color=C['xgb'])
ax.axhline(0,color='#8B949E',lw=1,ls=':')
ax.axvline(THRESHOLD,color=C['warn'],lw=1.5,ls='--',label=f'Padrão ({THRESHOLD})')
ax.axvline(best_t,color=C['accent'],lw=2,ls='-',label=f'Ótimo ({best_t:.2f})')
ax.scatter([best_t],[max(profits)/1000],color=C['accent'],s=100,zorder=5,marker='*')
ax.set_xlabel('Threshold'); ax.set_ylabel('Lucro ×1000 R$')
ax.set_title(f'Curva Lucro × Threshold — {best_name} (GMSC)',
             fontsize=13,fontweight='bold',color='#E6EDF3')
ax.legend(fontsize=10,framealpha=0.3); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print(f'\n  Threshold padrão ({THRESHOLD}) → R${simulate_business(best_prob,y_test,THRESHOLD)["lucro"]:,.0f}')
print(f'  Threshold ótimo  ({best_t:.3f}) → R${max(profits):,.0f}')

---
## 11. 🔲 Matrizes de Confusão

In [ ]:
fig,axes = plt.subplots(2,3,figsize=(16,9)); fig.patch.set_facecolor('#0F1117')

for col_idx,r in enumerate(all_results):
    for row_idx,(normalize,sfx) in enumerate([(None,'Absoluta'),('true','Normalizada')]):
        ax = axes[row_idx][col_idx]
        cm = confusion_matrix(y_test,r['y_pred'],normalize=normalize)
        ConfusionMatrixDisplay(cm,display_labels=['Adimpl.','Inadimpl.']).plot(
            ax=ax,colorbar=False,cmap='Blues')
        ax.set_title(f'{r["name"]}\n({sfx})',fontsize=10,color='#E6EDF3',pad=8)
        if normalize:
            for t in ax.texts: t.set_text(f'{float(t.get_text()):.1%}'); t.set_fontsize(13)

plt.suptitle('Matrizes de Confusão — GMSC',fontsize=14,color=C['accent'],fontweight='bold')
plt.tight_layout(); plt.show()

---
## 12. 🏁 Conclusão

In [ ]:
print('╔═══════════════════════════════════════════════════════════════╗')
print('║      CREDIT RISK ML — GMSC — RESULTADOS FINAIS              ║')
print('╠═══════════════════════════════════════════════════════════════╣')
print(f'║  Dataset : Give Me Some Credit ({len(df):,} registros)            ║')
print(f'║  Target  : SeriousDlqin2yrs  ({df["default"].mean():.1%} de inadimplência)    ║')
print(f'║  Features: {len(FEATURES)} ({len([c for c in FEATURES if c not in ["default"]])-7} base + 7 derivadas)                       ║')
print('║                                                               ║')
print('║  MODELO             AUC      F1     RECALL  PRECISION        ║')
for r in all_results:
    star = '⭐' if r['name']==best_name else '  '
    print(f'║  {star} {r["name"]:<18} {r["auc"]:.4f}  {r["f1"]:.4f}  {r["recall"]:.4f}  {r["precision"]:.4f}  ║')
print('║                                                               ║')
print(f'║  Threshold padrão : {THRESHOLD}                                    ║')
print(f'║  Threshold ótimo  : {best_t:.3f}                                ║')
print('╚═══════════════════════════════════════════════════════════════╝')